# Generation and Analysis of k-Path Graphs

Notebook definitivo para execução do pipeline principal usando os módulos de `src`.

Este arquivo **não** contém benchmarks/testes nem implementações duplicadas do notebook original.

## 1) Importações do pipeline modular
As funções abaixo já foram refatoradas e centralizadas em `src`.

In [3]:
# Ajuste de caminho: permite importar `src` mesmo com kernel iniciado em `notebooks/`
import sys
from pathlib import Path
import time

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root ativo: {project_root}")

# Funções de geração (stream) e fórmula T_n_k
from src.generators import gerarSequenciaN_stream, T_n_k

# Funções de conversão para Graph6 em modo stream
from src.converters import generateOneG6_stream

# Funções de conectividade algébrica em modo stream
from src.analyzers import verify_algebraic_connectivity_one_stream, TableMaxCA, indexCA

Project root ativo: /home/rafael/Documentos/Doutorado


## 2) Configuração de execução (plano completo)
Defina as faixas por `k`. O notebook executa de forma iterativa (stream), sem carregar tudo em memória.

In [4]:
# Plano definido
PLAN = {
    2: (6, 33),
    3: (8, 25),
    4: (10, 23),
}

base_seq = Path("data/sequences")
base_g6 = Path("data/g6")
base_ca = Path("data/algebraic_connectivity")

base_seq.mkdir(parents=True, exist_ok=True)
base_g6.mkdir(parents=True, exist_ok=True)
base_ca.mkdir(parents=True, exist_ok=True)

print("Plano ativo:")
for k, (n_inicio, n_fim) in PLAN.items():
    print(f"- k={k}: n={n_inicio}..{n_fim}")

Plano ativo:
- k=2: n=6..33
- k=3: n=8..25
- k=4: n=10..23


## 3) Geração de sequências (stream)
Gera os arquivos `.txt` faltantes, iterando por `k` e `n` no plano.

In [5]:
for k, (n_inicio, n_fim) in PLAN.items():
    seq_dir = base_seq / f"{k}_caminhos"
    seq_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[SEQUENCES] k={k} | n={n_inicio}..{n_fim}")
    for n in range(n_inicio, n_fim + 1):
        expected = int(T_n_k(n, k))
        seq_file = seq_dir / f"{k}_caminhos_n_{n}_T_{expected}.txt"

        if seq_file.exists():
            print(f"  [SKIP] n={n} (já existe)")
            continue

        t0 = time.time()
        info = gerarSequenciaN_stream(n, k, pasta_destino=seq_dir)
        dt = time.time() - t0
        print(
            f"  [OK] n={n} | expected={info['expected']} observed={info['observed']} ",
            f"| match={info['match']} | t={dt:.2f}s",
        )


[SEQUENCES] k=2 | n=6..33
  [OK] n=6 | expected=2 observed=2  | match=True | t=0.00s
  [OK] n=7 | expected=3 observed=3  | match=True | t=0.00s
  [OK] n=8 | expected=6 observed=6  | match=True | t=0.00s
  [OK] n=9 | expected=10 observed=10  | match=True | t=0.00s
  [OK] n=10 | expected=20 observed=20  | match=True | t=0.00s
  [OK] n=11 | expected=36 observed=36  | match=True | t=0.00s
  [OK] n=12 | expected=72 observed=72  | match=True | t=0.00s
  [OK] n=13 | expected=136 observed=136  | match=True | t=0.00s
  [OK] n=14 | expected=272 observed=272  | match=True | t=0.00s
  [OK] n=15 | expected=528 observed=528  | match=True | t=0.01s
  [OK] n=16 | expected=1056 observed=1056  | match=True | t=0.01s
  [OK] n=17 | expected=2080 observed=2080  | match=True | t=0.03s
  [OK] n=18 | expected=4160 observed=4160  | match=True | t=0.05s
  [OK] n=19 | expected=8256 observed=8256  | match=True | t=0.10s
  [OK] n=20 | expected=16512 observed=16512  | match=True | t=0.21s
  [OK] n=21 | expected=32

## 4) Conversão para Graph6 (stream)
Converte cada arquivo de sequência para `.g6` de forma iterativa.

In [6]:
for k, (n_inicio, n_fim) in PLAN.items():
    seq_dir = base_seq / f"{k}_caminhos"
    g6_dir = base_g6 / f"{k}_caminhos_g6"
    g6_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[G6] k={k} | n={n_inicio}..{n_fim}")
    for n in range(n_inicio, n_fim + 1):
        expected = int(T_n_k(n, k))
        g6_file = g6_dir / f"{k}_caminhos_n_{n}_T_{expected}.g6"

        if g6_file.exists():
            print(f"  [SKIP] n={n} (já existe)")
            continue

        t0 = time.time()
        info = generateOneG6_stream(n, k, pasta_origem=seq_dir, pasta_destino=g6_dir)
        dt = time.time() - t0
        print(
            f"  [OK] n={n} | expected={info['expected']} observed={info['observed']} ",
            f"| match={info['match']} | t={dt:.2f}s",
        )


[G6] k=2 | n=6..33
  [OK] n=6 | expected=2 observed=2  | match=True | t=0.15s
  [OK] n=7 | expected=3 observed=3  | match=True | t=0.00s
  [OK] n=8 | expected=6 observed=6  | match=True | t=0.00s
  [OK] n=9 | expected=10 observed=10  | match=True | t=0.00s
  [OK] n=10 | expected=20 observed=20  | match=True | t=0.00s
  [OK] n=11 | expected=36 observed=36  | match=True | t=0.01s
  [OK] n=12 | expected=72 observed=72  | match=True | t=0.01s
  [OK] n=13 | expected=136 observed=136  | match=True | t=0.03s
  [OK] n=14 | expected=272 observed=272  | match=True | t=0.06s
  [OK] n=15 | expected=528 observed=528  | match=True | t=0.12s
  [OK] n=16 | expected=1056 observed=1056  | match=True | t=0.28s
  [OK] n=17 | expected=2080 observed=2080  | match=True | t=0.62s
  [OK] n=18 | expected=4160 observed=4160  | match=True | t=1.29s
  [OK] n=19 | expected=8256 observed=8256  | match=True | t=2.67s
  [OK] n=20 | expected=16512 observed=16512  | match=True | t=5.54s
  [OK] n=21 | expected=32896 obs

KeyboardInterrupt: 

## 5) Conectividade algébrica (stream)
Calcula a conectividade para cada `.g6` sem carregar todos os grafos em memória e atualiza `k_CA_lista.txt`.

In [ ]:
for k, (n_inicio, n_fim) in PLAN.items():
    g6_dir = base_g6 / f"{k}_caminhos_g6"
    ca_dir = base_ca / f"CA_{k}_path_graph"
    ca_list_path = base_ca / f"{k}_CA_lista.txt"
    ca_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[CA] k={k} | n={n_inicio}..{n_fim}")
    lines = []

    for n in range(n_inicio, n_fim + 1):
        ca_file = ca_dir / f"{k}_CAs_n_{n}.txt"

        if ca_file.exists():
            print(f"  [SKIP] n={n} (já existe)")
        else:
            t0 = time.time()
            info = verify_algebraic_connectivity_one_stream(n, k, g6_base=g6_dir, ca_base=ca_dir)
            dt = time.time() - t0
            print(
                f"  [OK] n={n} | expected={info['expected']} observed={info['observed']} ",
                f"| match={info['match']} | t={dt:.2f}s",
            )

        # Reconstrói a linha de resumo a partir do arquivo CA
        vals = []
        with open(ca_file, "r", encoding="utf-8") as f:
            for raw in f:
                s = raw.strip()
                if s:
                    vals.append(float(s))

        if vals:
            max_ca = max(vals)
            min_ca = min(vals)
            pos_max = vals.index(max_ca)
            pos_min = vals.index(min_ca)
            lines.append(
                f"N({n}) 3_max_CA({max_ca}) pos_elemento_max_CA({pos_max}) | ",
                f"min_CA({min_ca}) pos_elemento_min_CA({pos_min})  \\n",
            )

    with open(ca_list_path, "w", encoding="utf-8") as f:
        for a, b in lines:
            f.write(a + b)

    print(f"[DONE] lista consolidada: {ca_list_path}")

## 6) Resumos opcionais
Gera tabelas de máximos e indexação de CA para cada `k` até o `n_fim` do plano.

In [ ]:
for k, (_, n_fim) in PLAN.items():
    ca_dir = base_ca / f"CA_{k}_path_graph"

    arquivo_max = TableMaxCA(n_fim, k, ca_base=ca_dir)
    arquivos_indexados = indexCA(n_fim, k, ca_base=ca_dir)

    print(f"\n[SUMMARY] k={k}")
    print(f"- Tabela de máximos: {arquivo_max}")
    print(f"- Arquivos indexados: {len(arquivos_indexados)}")